# Notebook 8: Experiment 4 — Multi-Stock Training (70/30)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on 3 stocks combined, predict on the remaining 1 stock.  
**Train/Test Split:** 70/30 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  

**Combinations:**
- Train TLKM+BBCA+ASII → Predict UNVR
- Train TLKM+BBCA+UNVR → Predict ASII
- Train TLKM+ASII+UNVR → Predict BBCA
- Train BBCA+ASII+UNVR → Predict TLKM


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

import plotly.graph_objects as go
from plotly.subplots import make_subplots

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.7
RATIO_LABEL = '70_30'
EXP_LABEL = f'Exp4_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 4 - Multi-Stock Training (70/30)")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 4 - Multi-Stock Training (70/30)


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


In [3]:
# Reload the module to get the latest fixes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
✓ Module reloaded successfully


## Define Training Combinations

In [4]:
# ============================================================
# MULTI-STOCK COMBINATIONS
# ============================================================
# Each entry: (training_stocks, target_stock)
combinations = []
for target in STOCKS:
    train_stocks = [s for s in STOCKS if s != target]
    combinations.append((train_stocks, target))
    print(f"  Train: {', '.join(train_stocks)} -> Predict: {target}")


  Train: BBCA, ASII, UNVR -> Predict: TLKM
  Train: TLKM, ASII, UNVR -> Predict: BBCA
  Train: TLKM, BBCA, UNVR -> Predict: ASII
  Train: TLKM, BBCA, ASII -> Predict: UNVR


## Run All Multi-Stock Experiments

In [5]:
# ============================================================
# EXPERIMENT 4: Multi-stock training
# ============================================================
all_results = []
all_predictions = {}

for train_stocks, target_stock in combinations:
    train_label = '+'.join(train_stocks)
    pair_key = (train_label, target_stock)
    
    print(f"\n{'#'*60}")
    print(f"# TRAIN: {train_label} -> TARGET: {target_stock}")
    print(f"{'#'*60}")
    
    # Prepare multi-stock data
    train_dfs = [daily_data[s] for s in train_stocks]
    test_df = daily_data[target_stock]
    
    X_train, y_train, X_test, y_test, test_dates = prepare_multi_stock_data(
        train_dfs, test_df,
        train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train (combined): {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[pair_key] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_train_{train_label}_target_{target_stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        result = {
            'Train_Stocks': train_label,
            'Target_Stock': target_stock,
            'Model': model_type,
            **metrics
        }
        all_results.append(result)
        all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
        
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, f'Train_{train_label}_Target_{target_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 4 (70/30) training complete!")



############################################################
# TRAIN: BBCA+ASII+UNVR -> TARGET: TLKM
############################################################
  X_train (combined): (11008, 1, 1), X_test: (1573, 1, 1)

Training BiLSTM for: Exp4_70_30_train_BBCA+ASII+UNVR_target_TLKM
  Train samples: 11008, Test samples: 1573
Epoch 1/100
154/155 [============================>.] - ETA: 0s - loss: 0.0112
Epoch 1: val_loss improved from inf to 0.00010, saving model to models/Exp4_70_30\Exp4_70_30_train_BBCA+ASII+UNVR_target_TLKM_BiLSTM_best.keras
155/155 [==============================] - 11s 21ms/step - loss: 0.0112 - val_loss: 9.7328e-05
Epoch 2/100
151/155 [============================>.] - ETA: 0s - loss: 4.9623e-04
Epoch 2: val_loss improved from 0.00010 to 0.00006, saving model to models/Exp4_70_30\Exp4_70_30_train_BBCA+ASII+UNVR_target_TLKM_BiLSTM_best.keras
155/155 [==============================] - 2s 12ms/step - loss: 4.9740e-04 - val_loss: 6.1671e-05
Epoch 3/100
151/155 [====

## Results Summary

In [6]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 4 - Multi-Stock Training (70/30)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 4 - Multi-Stock Training (70/30)
  Train_Stocks Target_Stock  Model         MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
BBCA+ASII+UNVR         TLKM BiLSTM   5175.0945  71.9381  57.2305    1.9383 0.975430            184.0         100
BBCA+ASII+UNVR         TLKM  BiGRU   3153.9227  56.1598  41.9654    1.4432 0.985026            170.9         100
BBCA+ASII+UNVR         TLKM   LSTM   4479.0525  66.9257  51.8505    1.7981 0.978734            122.9         100
BBCA+ASII+UNVR         TLKM    GRU   4186.8335  64.7057  49.7486    1.7147 0.980122            117.7         100
TLKM+ASII+UNVR         BBCA BiLSTM  97614.4008 312.4330 256.1928    3.2615 0.960920            188.1         100
TLKM+ASII+UNVR         BBCA  BiGRU  53639.4244 231.6019 185.3894    2.3833 0.978525            175.4         100
TLKM+ASII+UNVR         BBCA   LSTM 108004.8773 328.6410 238.8338    2.8963 0.956760            127.8         100
TLKM+ASII+UNVR         BBCA    GRU  85695.9188 29

## Visualizations

In [ ]:
# ============================================================
# COMPREHENSIVE INTERACTIVE RESULTS DASHBOARDS
# ============================================================

print("\n" + "="*70)
print("  GENERATING COMPREHENSIVE INTERACTIVE DASHBOARDS")
print("="*70 + "\n")

# 1. Metrics Heatmaps by Target Stock
print("1. Generating Metrics Heatmaps (RMSE, MAE, R²) by Target Stock...\n")
create_interactive_experiment4_dashboard(
    all_predictions, results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)

# 2. Model Comparison for Each Target Stock
print("\n2. Generating Model Performance Comparison for Each Target Stock...\n")
for target_stock in STOCKS:
    fig, html_file = create_interactive_model_comparison_by_target(
        results_df, target_stock, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
    )
    if fig is not None:
        print(f"   ✓ {target_stock}: {html_file}")
        fig.show()

# 3. Overall Metrics Comparison
print("\n3. Generating Overall Metrics Comparison...\n")
fig_metrics, html_metrics = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2', 'MAPE (%)'] if 'MAPE (%)' in results_df.columns else ['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Metrics Comparison: {html_metrics}")
fig_metrics.show()

# 4. Model Performance Radar Chart
print("\n4. Generating Model Performance Radar Chart...\n")
fig_radar, html_radar = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Radar Chart: {html_radar}")
fig_radar.show()

print("\n" + "="*70)
print("  ✓ ALL INTERACTIVE VISUALIZATIONS GENERATED SUCCESSFULLY!")
print("="*70)
print(f"\nAll visualizations saved to: figures/{EXP_LABEL}/")


## Interactive Results Visualizations

In [ ]:
# ============================================================
# INTERACTIVE VISUALIZATIONS - ACTUAL VS PREDICTED
# ============================================================
print("\n" + "="*70)
print("  GENERATING INTERACTIVE ACTUAL vs PREDICTED COMPARISONS")
print("="*70 + "\n")

for (train_label, target_stock), preds_dict in all_predictions.items():
    y_true = preds_dict[MODEL_TYPES[0]][0]  # Get actual prices (same for all models)
    test_dates = preds_dict[MODEL_TYPES[0]][2]  # Get dates
    
    # Create predictions dict for visualization
    predictions = {}
    for model_type in MODEL_TYPES:
        if model_type in preds_dict:
            predictions[model_type] = preds_dict[model_type][1]  # Get predicted prices
    
    # Create interactive actual vs predicted plot
    fig, html_file = create_interactive_actual_vs_predicted_exp4(
        test_dates, y_true, predictions,
        train_label, target_stock,
        EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
    )
    
    print(f"  ✓ {target_stock}: Train({train_label}) → {html_file}")
    fig.show()

print("\n✓ All actual vs predicted comparisons generated!")


In [ ]:
# ============================================================
# SUMMARY OF ALL INTERACTIVE HTML FILES
# ============================================================
import os
import glob

print("\n" + "="*70)
print("  INTERACTIVE HTML FILES GENERATED")
print("="*70 + "\n")

html_dir = f'figures/{EXP_LABEL}'
html_files = sorted(glob.glob(os.path.join(html_dir, '*.html')))

print(f"Total interactive visualizations: {len(html_files)}\n")
print("Files created:\n")
for i, html_file in enumerate(html_files, 1):
    filename = os.path.basename(html_file)
    file_size_kb = os.path.getsize(html_file) / 1024
    print(f"  {i:2d}. {filename}")
    print(f"      Size: {file_size_kb:.1f} KB")
    print(f"      Path: {html_file}\n")

print("="*70)
print(f"Directory: {os.path.abspath(html_dir)}")
print("="*70)


In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER TARGET STOCK (by RMSE)")
print("="*70)
for target in STOCKS:
    target_data = results_df[results_df['Target_Stock'] == target]
    if target_data.empty:
        continue
    best_idx = target_data['RMSE'].idxmin()
    best = target_data.loc[best_idx]
    print(f"  Target {target}: Trained on {best['Train_Stocks']} + {best['Model']} "
          f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")


## Case-by-Case Interactive Actual vs Predicted
One interactive Plotly chart per notebook with a dropdown that walks through every test case. Each case shows the Actual price (red) plus all four model predictions, with a metrics panel (MSE, RMSE, MAE, MAPE, R²) for that case.

In [ ]:
# ============================================================
# CASE-BY-CASE INTERACTIVE ACTUAL VS PREDICTED (Experiment 4)
# ============================================================
# Cases follow the test plan: train on 3 stocks combined -> predict the 4th.
from collections import OrderedDict

cases_dict = OrderedDict()
for (train_label, target_stock), preds_dict in all_predictions.items():
    available_mts = [mt for mt in MODEL_TYPES if mt in preds_dict]
    if not available_mts:
        continue
    y_true, _, dates = preds_dict[available_mts[0]]
    predictions = {mt: preds_dict[mt][1] for mt in available_mts}

    metrics = {}
    for mt in MODEL_TYPES:
        row = results_df[(results_df['Train_Stocks'] == train_label) &
                         (results_df['Target_Stock'] == target_stock) &
                         (results_df['Model']        == mt)]
        if not row.empty:
            r = row.iloc[0]
            metrics[mt] = {
                'MSE'      : r.get('MSE'),
                'RMSE'     : r.get('RMSE'),
                'MAE'      : r.get('MAE'),
                'MAPE (%)' : r.get('MAPE (%)'),
                'R2'       : r.get('R2'),
            }

    case_label = f'→{target_stock}'
    cases_dict[case_label] = {
        'description' : f'Train [{train_label}] Daily → Predict {target_stock} Daily',
        'dates'       : dates,
        'y_true'      : y_true,
        'predictions' : predictions,
        'metrics'     : metrics,
    }

ratio_pretty = RATIO_LABEL.replace('_', '/')
fig_case, html_case = create_case_by_case_actual_vs_predicted(
    cases_dict,
    experiment_title=f'Experiment 4: Multi-Stock Training ({ratio_pretty})',
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}',
)
print(f"✓ Case-by-case visualization saved: {html_case}")
print(f"  Cases ({len(cases_dict)}): {list(cases_dict.keys())}")
fig_case.show()
